# QA Sub-Saharan African Bat Occurrence Records with Taxonomy, Locality, Coordinates, and Collection Metadata Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.yade-g5zy/fair2.json`

In this notebook, we:
- load the Croissant metadata,
- inspect dataset-level information,
- review available record sets using their `@id` values,
- attempt to load records into pandas DataFrames,
- apply light exploratory processing when tabular records are available,
- and visualize high-level dataset metadata.

> Important: all dataset entities should be referenced by their `@id` values wherever available. This notebook uses variables to keep those references explicit and reusable.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`. We keep the Croissant URL in a variable so the notebook is easy to reuse.

In [ ]:
import json
from collections import Counter

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.yade-g5zy/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata

print('Dataset loaded successfully.')
print(f'Name: {metadata.name}')
print(f'Description: {metadata.description}')

In [ ]:
# Convert metadata to JSON-like structure for easier display when needed
metadata_json = metadata.to_json()

dataset_summary = {
    '@id': metadata_json.get('@id'),
    'name': metadata_json.get('name'),
    'identifier': metadata_json.get('identifier'),
    'version': metadata_json.get('version'),
    'license': metadata_json.get('license'),
    'datePublished': metadata_json.get('datePublished'),
    'temporalCoverage': metadata_json.get('temporalCoverage'),
    'spatialCoverage': metadata_json.get('spatialCoverage'),
    'conformsTo': metadata_json.get('conformsTo')
}

pd.DataFrame([dataset_summary]).T.rename(columns={0: 'value'})

## 2. Data Overview
Review the available Croissant entities, especially record sets and their `@id` values.

This dataset package includes rich dataset-level metadata. Some Croissant packages also expose one or more record sets for row-wise data access. We inspect the metadata first and then check which record sets are available.

In [ ]:
# High-level keys present in the metadata JSON
sorted(metadata_json.keys())

In [ ]:
# Inspect record sets using their @id values
record_sets = metadata_json.get('recordSet', [])
print(f'Number of record sets declared: {len(record_sets)}')

record_set_rows = []
for rs in record_sets:
    record_set_rows.append({
        'record_set_@id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description')
    })

pd.DataFrame(record_set_rows) if record_set_rows else pd.DataFrame(columns=['record_set_@id', 'name', 'description'])

The supplied FAIR^2 package indicates that `recordSet` is currently empty. That means there may be no row-level tabular record sets exposed directly through the Croissant metadata at this URL.

Still, it is useful to inspect the other dataset entities that do have `@id` values, such as authors, citations, and distributions.

In [ ]:
# Inspect distribution entities by @id
distribution_rows = []
for dist in metadata_json.get('distribution', []):
    distribution_rows.append({
        'distribution_@id': dist.get('@id'),
        'name': dist.get('name'),
        'contentUrl': dist.get('contentUrl'),
        'encodingFormat': dist.get('encodingFormat')
    })

pd.DataFrame(distribution_rows) if distribution_rows else pd.DataFrame(columns=['distribution_@id', 'name', 'contentUrl', 'encodingFormat'])

In [ ]:
# Inspect author entity references by @id
author_rows = []
for author in metadata_json.get('author', []):
    author_rows.append({'author_@id': author.get('@id')})

pd.DataFrame(author_rows) if author_rows else pd.DataFrame(columns=['author_@id'])

In [ ]:
# Inspect citation entity references by @id
citation_rows = []
for citation in metadata_json.get('citation', []):
    citation_rows.append({'citation_@id': citation.get('@id')})

citations_df = pd.DataFrame(citation_rows) if citation_rows else pd.DataFrame(columns=['citation_@id'])
citations_df.head(10)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Because this dataset currently declares no record sets, the extraction workflow below is written defensively. If record sets are added in a future version, the same pattern will work by referencing each record set's `@id`.

In [ ]:
# Extract data from each record set, using @id references when available
record_set_ids = [rs.get('@id') for rs in record_sets if rs.get('@id')]
dataframes = {}

if not record_set_ids:
    print('No record sets are declared in this Croissant metadata, so no row-level records can be loaded with dataset.records(...).')
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} rows from record set: {record_set_id}')

In [ ]:
# Example preview if record sets become available in a future release
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print('First record set @id:', first_record_set_id)
    print('Columns:', dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No DataFrame preview available because no record sets were loaded.')

Since no row-level record sets are available, we can still build useful analysis tables from dataset metadata itself. This is often helpful for understanding documentation completeness and the structure of a Croissant package.

In [ ]:
# Build a metadata-centric analysis table
metadata_analysis_rows = [
    {'field_@id': 'schema:keywords', 'value': kw, 'value_type': 'keyword'}
    for kw in metadata_json.get('keywords', [])
]

metadata_analysis_rows += [
    {'field_@id': 'rai:dataBiases', 'value': item, 'value_type': 'data_bias'}
    for item in metadata_json.get('dataBiases', [])
]

metadata_analysis_rows += [
    {'field_@id': 'rai:dataLimitations', 'value': item, 'value_type': 'data_limitation'}
    for item in metadata_json.get('dataLimitations', [])
]

metadata_analysis_rows += [
    {'field_@id': 'rai:dataUseCases', 'value': item, 'value_type': 'data_use_case'}
    for item in metadata_json.get('dataUseCases', [])
]

metadata_df = pd.DataFrame(metadata_analysis_rows)
metadata_df.head(10)

## 4. Exploratory Data Analysis (EDA)
The template asks for common processing steps such as filtering, normalization, and grouping.

Because no tabular record set is declared, we apply those ideas to metadata-derived analysis tables instead. We use the length of each metadata text entry as a simple numeric feature for exploration.

In [ ]:
if not metadata_df.empty:
    analysis_df = metadata_df.copy()
    numeric_field_id = 'derived:text_length'
    group_field_id = 'value_type'

    analysis_df[numeric_field_id] = analysis_df['value'].astype(str).str.len()
    threshold = 40

    filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()
    print(f'Filtered rows with {numeric_field_id} > {threshold}: {len(filtered_df)}')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field_id].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f'{numeric_field_id}_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / std_value
    else:
        filtered_df[f'{numeric_field_id}_normalized'] = 0.0

    print(f'Normalized {numeric_field_id} for filtered rows:')
    display(filtered_df[[group_field_id, 'value', numeric_field_id, f'{numeric_field_id}_normalized']].head())

    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'median', 'max']).sort_values('count', ascending=False)
    print(f'Grouped data by {group_field_id}:')
    display(grouped_df)
else:
    print('Metadata-derived analysis table is empty.')

In [ ]:
# Additional overview of keyword lengths
keywords = metadata_json.get('keywords', [])
keywords_df = pd.DataFrame({'field_@id': 'schema:keywords', 'keyword': keywords})

if not keywords_df.empty:
    keywords_df['derived:keyword_length'] = keywords_df['keyword'].str.len()
    display(keywords_df.sort_values('derived:keyword_length', ascending=False).head(10))
else:
    print('No keywords available.')

## 5. Visualization
Visualize the distributions of metadata-derived features and summarize the structure of the dataset package.

In [ ]:
# Bar chart of metadata entry counts by type
if not metadata_df.empty:
    plt.figure(figsize=(8, 4))
    order = metadata_df['value_type'].value_counts().index
    sns.countplot(data=metadata_df, y='value_type', order=order, palette='viridis')
    plt.title('Counts of Metadata Entries by Type')
    plt.xlabel('Count')
    plt.ylabel('Metadata entry type')
    plt.tight_layout()
    plt.show()
else:
    print('No metadata entries available for plotting.')

In [ ]:
# Distribution of text lengths for metadata-derived entries
if not metadata_df.empty:
    plot_df = metadata_df.copy()
    plot_df['derived:text_length'] = plot_df['value'].astype(str).str.len()

    plt.figure(figsize=(8, 4))
    sns.histplot(plot_df['derived:text_length'], bins=15, kde=True, color='steelblue')
    plt.title('Distribution of Metadata Text Lengths')
    plt.xlabel('Text length')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print('No text lengths available for plotting.')

In [ ]:
# Summary chart of top keyword lengths
if not keywords_df.empty:
    top_keywords_df = keywords_df.sort_values('derived:keyword_length', ascending=False).head(10)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_keywords_df, x='derived:keyword_length', y='keyword', palette='mako')
    plt.title('Top 10 Longest Keywords')
    plt.xlabel('Keyword length')
    plt.ylabel('Keyword')
    plt.tight_layout()
    plt.show()
else:
    print('No keyword visualization available.')

In [ ]:
# Compact package structure summary
package_structure_df = pd.DataFrame([
    {'entity_group_@id': 'cr:RecordSet', 'count': len(metadata_json.get('recordSet', []))},
    {'entity_group_@id': 'schema:DataDownload', 'count': len(metadata_json.get('distribution', []))},
    {'entity_group_@id': 'schema:Person', 'count': len(metadata_json.get('author', []))},
    {'entity_group_@id': 'schema:CreativeWork', 'count': len(metadata_json.get('citation', []))},
    {'entity_group_@id': 'schema:DefinedTermSet', 'count': len(metadata_json.get('keywords', []))}
])

plt.figure(figsize=(8, 4))
sns.barplot(data=package_structure_df, x='count', y='entity_group_@id', palette='crest')
plt.title('Croissant Package Structure Summary')
plt.xlabel('Count')
plt.ylabel('Entity group @id')
plt.tight_layout()
plt.show()

package_structure_df

## 6. Conclusion
This exploration shows that the FAIR^2 Croissant package contains rich dataset-level metadata describing a curated bat occurrence resource for sub-Saharan Africa.

Key observations:
- The dataset metadata loads successfully with `mlcroissant`.
- The package clearly documents description, licensing, temporal and spatial scope, biases, limitations, and use cases.
- In the supplied version, no `cr:RecordSet` entities are declared, so row-level records are not directly accessible through `dataset.records(record_set=...)`.
- Even without record sets, metadata itself can still be explored, summarized, and visualized.

If a future version of this Croissant package exposes one or more record sets, the extraction section of this notebook can be reused directly by supplying the relevant record set `@id` values.